# Data Cleaning & Preprocessing

This notebook focuses on preparing the dataset for risk analysis and dashboard development.

Objectives:
- Convert date columns into datetime format.
- Handle missing values.
- Create business-oriented risk indicators.
- Generate a clean dataset for further analysis.

In [3]:
import pandas as pd
df = pd.read_csv('financial_loan.csv')
df.head()

,id,address_state,application_type,emp_length,emp_title,grade,home_ownership,issue_date,last_credit_pull_date,last_payment_date,...,sub_grade,term,verification_status,annual_income,dti,installment,int_rate,loan_amount,total_acc,total_payment
0,1077430,GA,INDIVIDUAL,< 1 year,Ryder,C,RENT,11-02-2021,13-09-2021,13-04-2021,...,C4,60 months,Source Verified,30000.0,0.0100,59.83,0.1527,2500,4,1009
1,1072053,CA,INDIVIDUAL,9 years,MKC Accounting,E,RENT,01-01-2021,14-12-2021,15-01-2021,...,E1,36 months,Source Verified,48000.0,0.0535,109.43,0.1864,3000,4,3939
2,1069243,CA,INDIVIDUAL,4 years,Chemat Technology Inc,C,RENT,05-01-2021,12-12-2021,09-01-2021,...,C5,36 months,Not Verified,50000.0,0.2088,421.65,0.1596,12000,11,3522
3,1041756,TX,INDIVIDUAL,< 1 year,barnes distribution,B,MORTGAGE,25-02-2021,12-12-2021,12-03-2021,...,B2,60 months,Source Verified,42000.0,0.0540,97.06,0.1065,4500,9,4911
4,1068350,IL,INDIVIDUAL,10+ years,J&J Steel Inc,A,MORTGAGE,01-01-2021,14-12-2021,15-01-2021,...,A1,36 months,Verified,83000.0,0.0231,106.53,0.0603,3500,28,3835


1. Date Conversion

In [8]:
df['issue_date'].dtypes

dtype('O')

In [10]:
date_cols = [col for col in df.columns if 'date' in col]
print(date_cols)

for col in date_cols:
    df[col] = pd.to_datetime(df[col],format = '%d-%m-%Y')

['issue_date', 'last_credit_pull_date', 'last_payment_date', 'next_payment_date']


In [13]:
df['issue_date'].dtypes

dtype('<M8[ns]')

2. Missing Value Treatment

The dataset contains only one column with missing values (`emp_title`).

Instead of removing observations, missing values are replaced with `"Unknown"` to preserve the entire portfolio for analysis.

In [16]:
df.isnull().sum().sort_values(ascending=False)

emp_title                1438
id                          0
purpose                     0
total_acc                   0
loan_amount                 0
int_rate                    0
installment                 0
dti                         0
annual_income               0
verification_status         0
term                        0
sub_grade                   0
member_id                   0
address_state               0
next_payment_date           0
loan_status                 0
last_payment_date           0
last_credit_pull_date       0
issue_date                  0
home_ownership              0
grade                       0
emp_length                  0
application_type            0
total_payment               0
dtype: int64

In [17]:
df['emp_title'] = df['emp_title'].fillna('Unknown')

In [18]:
df.isnull().sum().sort_values(ascending=False)

id                       0
address_state            0
total_acc                0
loan_amount              0
int_rate                 0
installment              0
dti                      0
annual_income            0
verification_status      0
term                     0
sub_grade                0
purpose                  0
member_id                0
next_payment_date        0
loan_status              0
last_payment_date        0
last_credit_pull_date    0
issue_date               0
home_ownership           0
grade                    0
emp_title                0
emp_length               0
application_type         0
total_payment            0
dtype: int64

3. Loan Classification

Loans are categorized into two groups:

- Good Loan: Fully Paid or Current
- Bad Loan: Charged Off

This classification simplifies portfolio monitoring and risk reporting.

In [34]:
df['loan_category'] = df['loan_status'].apply(lambda x : 'Bad Loan'  
                                              if x == 'Charged Off'
                                              else 'Good Loan')

df['loan_category'].value_counts()

loan_category
Good Loan    33243
Bad Loan      5333
Name: count, dtype: int64

4. Default Indicator

A binary default flag is created to facilitate risk calculations and dashboard metrics.

- 1 = Default (Charged Off)
- 0 = Non-default

In [37]:
df['loan_default'] = df['loan_category'].apply(lambda x : 1
                                              if x == 'Bad Loan'
                                              else 0)

df[['loan_category','loan_default']].head()


,loan_category,loan_default
0,Bad Loan,1
1,Good Loan,0
2,Bad Loan,1
3,Good Loan,0
4,Good Loan,0


5. Portfolio Default Rate

The default rate is a key indicator used to assess the quality of a loan portfolio.

Since `loan_default` is encoded as a binary variable:

- 1 = Defaulted loan (Bad Loan)
- 0 = Non-defaulted loan (Good Loan)

The average value of this column directly represents the proportion of defaulted loans in the portfolio.

For example:

(0 + 0 + 1 + 0 + 1) / 5 = 0.40 = 40%

Therefore, the `mean()` function provides an efficient way to calculate the portfolio default rate.

In [48]:
default_rate = df['loan_default'].mean()
print(f"Portfolio Default Rate: {default_rate:.2%}") #ty le vo no cua toan bo doanh muc cho vay (giu 2 so sau %) 

Portfolio Default Rate: 13.82%


6. Time Feature Engineering

Additional time-based features are created to support temporal analysis and dashboard reporting (PowerBI).

In [59]:
df['issue_year'] = df['issue_date'].dt.year
df['issue_month'] = df['issue_date'].dt.month
df['issue_month_name'] = df['issue_date'].dt.month_name() 
df['issue_quarter'] = df['issue_date'].dt.quarter

issue_cols = [col for col in df.columns if 'issue' in col]

df[issue_cols].head()

,issue_date,issue_year,issue_month,issue_month_name,issue_quarter
0,2021-02-11,2021,2,February,1
1,2021-01-01,2021,1,January,1
2,2021-01-05,2021,1,January,1
3,2021-02-25,2021,2,February,1
4,2021-01-01,2021,1,January,1
